# Train the MLBB vision detector (free, in Google Colab)

Makes `mlbb_detector.tflite` + `labels.txt` for the MLBB Research app (Model page → *Choose detector file*).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. Make a free account at roboflow.com and copy your **API key** (Settings → API Keys). You type it into the box below; it is never saved in this notebook.
3. Default dataset: **Mobile Legends Minimap Dataset** by *gladi* on Roboflow Universe, licence **CC BY 4.0** (credit the author if you share the model).

Run each cell from top to bottom (tap ▶). Training takes about 10–20 minutes.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Settings
Change these only if you want a different Roboflow dataset. For a full-screen dataset, also set the app's *Detector looks at* to *the whole screen*.

In [ ]:
WORKSPACE = "gladi"                            # Roboflow workspace (from the dataset URL)
PROJECT   = "mobile-legends-minimap-dataset"   # Roboflow project (from the dataset URL)
VERSION   = 2                                  # dataset version number
EPOCHS    = 80                                 # more = better but slower
IMGSZ     = 320                                # 320 is fast on phones; 640 is more accurate

## 2. Download the dataset

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Paste your Roboflow API key and press Enter: "))
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print("Downloaded to", dataset.location)

## 3. Train a small YOLOv8 model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # 'nano' = smallest and fastest, good for phones
model.train(data=f"{dataset.location}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20)
best_path = model.trainer.best
print("Best weights:", best_path)

## 4. Export to LiteRT (.tflite) and download

In [ ]:
import glob, os, shutil, yaml

best = YOLO(best_path)
# Newer Ultralytics calls this format "litert"; older versions call it "tflite"
try:
    out = best.export(format="litert", imgsz=IMGSZ)
except Exception as e:
    print("litert export not available here, using tflite:", e)
    out = best.export(format="tflite", imgsz=IMGSZ)
print("Export result:", out)

# find the .tflite wherever Ultralytics saved it
weights_dir = os.path.dirname(str(best_path))
found = []
if out and str(out).endswith(".tflite"):
    found.append(str(out))
found += sorted(glob.glob(weights_dir + "/**/*.tflite", recursive=True))
found += sorted(glob.glob("/content/**/*.tflite", recursive=True))
found = [f for f in found if os.path.exists(f) and not f.endswith("mlbb_detector.tflite")]
if not found:
    print("No .tflite found. Files next to best.pt:")
    for root, dirs, fs in os.walk(weights_dir):
        for f in fs:
            print(os.path.join(root, f))
    raise SystemExit("Send this list to Claude.")
float32 = [f for f in found if "float32" in f]
pick = (float32 or found)[0]
shutil.copy(pick, "/content/mlbb_detector.tflite")
print("Using:", pick, round(os.path.getsize(pick) / 1e6, 1), "MB")

names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
open("/content/labels.txt", "w").write(",".join(names))
print("Classes (paste into the app if they differ from the default):")
print(",".join(names))

## 5. Download the files
On a phone the direct download often does nothing. If so, run the **Google Drive** cell instead: the files then appear in your Google Drive (My Drive), and the Drive app can save them to the phone.

In [ ]:
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/mlbb_detector.tflite")
files.download("/content/labels.txt")

In [ ]:
# B. Save to Google Drive (most reliable on a phone; asks you to allow Drive access)
from google.colab import drive
drive.mount("/content/drive")
shutil.copy("/content/mlbb_detector.tflite", "/content/drive/MyDrive/mlbb_detector.tflite")
shutil.copy("/content/labels.txt", "/content/drive/MyDrive/mlbb_labels.txt")
print("Saved to Google Drive > My Drive: mlbb_detector.tflite and mlbb_labels.txt")

## 6. Put it in the app
1. Copy `mlbb_detector.tflite` to your phone (it downloads straight to the phone if you run Colab there).
2. App → **Model** page → *Choose detector file (.tflite)*.
3. If the class list printed above differs from the app's default, paste it into *Class names*.
4. Home page preview shows the detections as cyan boxes.